# 第七课：图方向诊断与多随机种子复核

第六课的关键结果是 Identity 优于 Real，而 Real 优于 Shuffled。本课不急于寻找 test MAE 最小的图，而是提出并检验两个可证伪解释：

1. 原图是有向的，当前消息传播方向可能与交通影响方向相反；
2. 邻居聚合可能造成过平滑，即使方向正确也不如仅保留自身时间模式。

流程分为探索（单种子、只用 validation 选择候选）和确认（三个预注册种子、最终只报告 test）。

## 0. 防止 test 驱动的模型选择

探索阶段可以比较多个图，但只能根据 validation 排序。你应在看到探索 test 表之前就写下确认候选；本 Notebook 默认确认 `identity`、`real`、`transpose`、`symmetric`。如果以后更改候选，应明确标注为新的探索实验，而不是把它与确认实验混在一起。

In [ ]:
from copy import deepcopy
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), '请从项目根目录或 notebooks 目录启动 notebook'
sys.path.insert(0, str(ROOT / 'src'))

from crosscity.config import load_config  # noqa: E402
from crosscity.data.dataset import build_data_bundle  # noqa: E402
from crosscity.data.graph import load_adjacency, normalize_adjacency  # noqa: E402
from crosscity.evaluation.metrics import horizon_metrics, masked_mae  # noqa: E402
from crosscity.models.stgcn import STGCN  # noqa: E402
from crosscity.utils import seed_everything  # noqa: E402

config = load_config(ROOT / 'configs/metr_la.yaml')
bundle = build_data_bundle(config.dataset)
raw_a = load_adjacency(config.dataset.adjacency_path)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| raw graph symmetric:', np.allclose(raw_a, raw_a.T))

## 1. 构造图策略

若模型计算 `A @ X`，第 `i` 行代表节点 `i` 聚合哪些 `j` 的特征。转置 `A.T` 会反转这种信息接收方向。对称图保留双向连接，但不应被解释为真实道路方向；它只是一个诊断条件。所有策略都使用相同的“恰好一个自环”归一化规则。

In [ ]:
def without_diagonal(matrix):
    result = matrix.copy()
    np.fill_diagonal(result, 0)
    return result

raw_no_loop = without_diagonal(raw_a)
real_a = normalize_adjacency(raw_no_loop)
transpose_a = normalize_adjacency(raw_no_loop.T)
symmetric_a = normalize_adjacency((raw_no_loop + raw_no_loop.T) / 2)
identity_a = torch.eye(raw_a.shape[0])

shuffle_generator = torch.Generator().manual_seed(42)
permutation = torch.randperm(raw_a.shape[0], generator=shuffle_generator)
shuffled_a = real_a[permutation][:, permutation]
graphs = {
    'identity': identity_a,
    'real': real_a,
    'transpose': transpose_a,
    'symmetric': symmetric_a,
    'shuffled': shuffled_a,
}

def graph_summary(adjacency):
    return {
        'nonzero': int((adjacency > 0).sum()),
        'symmetric': bool(torch.allclose(adjacency, adjacency.T)),
        'row-sum min': adjacency.sum(1).min().item(),
        'row-sum max': adjacency.sum(1).max().item(),
    }

pd.DataFrame({name: graph_summary(graph) for name, graph in graphs.items()}).T.round(3)

注意：对称归一化不要求每行和为 1。`identity` 没有邻居边；`real` 与 `transpose` 的非零数量相同但边方向相反；`shuffled` 与 `real` 的数值分布相同但节点语义错位。

## 2. 可复用的公平训练函数

每一次训练都：重设随机种子、从同一初始化加载、用同一批次顺序训练，并只以 validation MAE 选择 checkpoint。训练和验证只用于模型选择；test 将在 checkpoint 固定后调用一次。

In [ ]:
FAST_MODE = True  # 首次运行保持 True；确认运行前改为 False
BATCH_SIZE = 64
MAX_EPOCHS = 6 if FAST_MODE else 50
PATIENCE = 3 if FAST_MODE else 8

def limited(dataset, maximum):
    return Subset(dataset, range(min(len(dataset), maximum)))

train_set = limited(bundle.train, 2048) if FAST_MODE else bundle.train
val_set = limited(bundle.val, 1024) if FAST_MODE else bundle.val
test_set = limited(bundle.test, 2048) if FAST_MODE else bundle.test
print('train / val / test windows:', len(train_set), len(val_set), len(test_set))

In [ ]:
@torch.no_grad()
def predict_metrics(model, dataset, adjacency):
    model.eval()
    predictions, targets, masks = [], [], []
    for xb, yb, mb in DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False):
        predictions.append(model(xb.to(device), adjacency).cpu())
        targets.append(yb)
        masks.append(mb)
    prediction = bundle.scaler.inverse_transform(torch.cat(predictions))
    target = bundle.scaler.inverse_transform(torch.cat(targets))
    return horizon_metrics(prediction, target, torch.cat(masks))

def overall_mae(model, dataset, adjacency):
    return predict_metrics(model, dataset, adjacency)['overall']['mae']

In [ ]:
def train_stgcn(graph_name, adjacency_cpu, seed, evaluate_test=False):
    seed_everything(seed)
    initial_model = STGCN(12, 12, hidden_dim=32)
    initial_state = deepcopy(initial_model.state_dict())
    model = STGCN(12, 12, hidden_dim=32).to(device)
    model.load_state_dict(initial_state)
    adjacency = adjacency_cpu.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loader_generator = torch.Generator().manual_seed(seed)
    best_val, best_state, stale = float('inf'), None, 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total, count = 0.0, 0
        loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator)
        for xb, yb, mb in loader:
            xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
            optimizer.zero_grad()
            loss = masked_mae(model(xb, adjacency), yb, mb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += loss.item() * int(mb.sum())
            count += int(mb.sum())
        val_mae = overall_mae(model, val_set, adjacency)
        history.append({'graph': graph_name, 'seed': seed, 'epoch': epoch, 'train_z_mae': total / count, 'val_mae': val_mae})
        if val_mae < best_val:
            best_val, best_state, stale = val_mae, deepcopy(model.state_dict()), 0
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    model.load_state_dict(best_state)
    validation_result = predict_metrics(model, val_set, adjacency)
    test_result = predict_metrics(model, test_set, adjacency) if evaluate_test else None
    return history, validation_result, test_result


这里每个图策略在**同一个 seed 内**从完全相同初始权重开始；不同 seed 则允许模型初始化和样本打乱变化，用来估计实验不确定性。

## 3. 探索阶段：一个种子，按 validation 排序

首次先用 `FAST_MODE=True` 跑通；要做真正的图方向诊断，将其改为 `False`，只重新运行本节。探索阶段**不会运行 test**，候选完全依据 validation。

In [ ]:
EXPLORATION_SEED = 42
explore_history, explore_val = [], {}
for name, adjacency in graphs.items():
    print('running:', name)
    history, val_result, _ = train_stgcn(name, adjacency, EXPLORATION_SEED)
    explore_history.extend(history)
    explore_val[name] = val_result



In [ ]:
exploration_validation = pd.Series({name: result['overall']['mae'] for name, result in explore_val.items()}).sort_values()
display(exploration_validation.to_frame('validation overall MAE (mph)').round(3))

history_frame = pd.DataFrame(explore_history)
plt.figure(figsize=(8, 4))
for name, group in history_frame.groupby('graph'):
    plt.plot(group.epoch, group.val_mae, marker='o', label=name)
plt.xlabel('epoch')
plt.ylabel('validation MAE (mph)')
plt.legend()
plt.title('Exploration: validation only')
plt.show()

在这里写下确认协议。推荐保留 `identity` 作为无空间信息基线，并确认 `real`、`transpose`、`symmetric`；`shuffled` 只作为负对照，通常不需要花费三种子预算。确认阶段之前不要改用任何 test 值替换候选。

In [ ]:
# 预注册：探索阶段未计算 test；确认运行前保持这一选择不变。
CONFIRM_GRAPHS = ['identity', 'real', 'transpose', 'symmetric']
CONFIRM_SEEDS = [42, 43, 44]
print('confirmation graphs:', CONFIRM_GRAPHS)
print('confirmation seeds :', CONFIRM_SEEDS)

## 4. 确认阶段：三个种子，最终 test 汇总

运行成本较高：FULL 模式会训练 4 个图策略 × 3 个种子 = 12 个模型。若 GPU 时间有限，先完成 Identity、Real、Transpose 三组；但请在报告中如实注明策略数量和种子数量。

In [ ]:
# 只有确认策略和种子固定后才运行此单元格。
confirm_history, confirm_validation, confirm_test = [], [], []
for seed in CONFIRM_SEEDS:
    for name in CONFIRM_GRAPHS:
        print(f'confirm | seed={seed} | graph={name}')
        history, val_result, test_result = train_stgcn(name, graphs[name], seed, evaluate_test=True)
        confirm_history.extend(history)
        confirm_validation.append({'seed': seed, 'graph': name, **val_result['overall']})
        for horizon, values in test_result.items():
            confirm_test.append({'seed': seed, 'graph': name, 'horizon': horizon, **values})

In [ ]:
confirm_test_frame = pd.DataFrame(confirm_test)
summary = confirm_test_frame.groupby(['graph', 'horizon'])[['mae', 'rmse', 'mape']].agg(['mean', 'std'])
summary.round(3)

In [ ]:
overall = confirm_test_frame.query("horizon == 'overall'")
overall_table = overall.pivot(index='seed', columns='graph', values='mae')
display(overall_table.round(3))
display(overall_table.agg(['mean', 'std']).T.round(3))

baseline = overall_table['identity']
relative_gain = overall_table.drop(columns='identity').rsub(baseline, axis=0).div(baseline, axis=0) * 100
relative_gain.columns = [f'{column} vs identity (%)' for column in relative_gain.columns]
relative_gain.agg(['mean', 'std']).T.round(2)

正增益代表该图优于 Identity，负增益代表更差。不要仅依据均值差异声称“显著”；三种子只能估计稳定性。若要做正式显著性检验，应保存每个测试窗口或每个传感器的误差，进行配对检验或 bootstrap。

In [ ]:
artifacts = ROOT / 'artifacts'
artifacts.mkdir(exist_ok=True)
pd.DataFrame(confirm_validation).to_csv(artifacts / 'notebook07_confirmation_validation.csv', index=False)
confirm_test_frame.to_csv(artifacts / 'notebook07_confirmation_test.csv', index=False)
summary.to_csv(artifacts / 'notebook07_confirmation_summary.csv')
print('saved confirmation CSV files to:', artifacts)

## 5. 如何解读所有可能结果

- **Identity 最好且稳定**：当前图聚合不适合该任务；时间模式是更强、更稳的归纳偏置。
- **Transpose 优于 Real**：原始图的消息方向可能与当前 `A @ X` 约定相反。
- **Symmetric 优于两种有向图**：方向信息在当前简化模型中不可靠，双向上下文更稳。
- **Real/Transpose/Symmetric 都优于 Identity**：空间关系本身带来稳定增益，再进入跨城市迁移实验。
- **所有图都不如 Identity、但 Real 优于 Shuffled**：真实图有信息，但单层静态聚合产生过平滑或不足以表达交通传播；这是合理的负结果。

## 验收问题

1. 为什么 `A.T` 是有效诊断，而不是任意调参？
2. 为什么对称图的表现好也不等于道路本来就是无向的？
3. 为什么候选图必须按 validation 选择，而不是按探索 test 表选择？
4. 为什么每个图策略在同一 seed 中必须从相同初始权重开始？
5. 若 Identity 稳定最优，跨城市实验中你会优先迁移什么组件？

完成后请带回探索 validation 排名、确认阶段每个图的 Overall MAE 均值±标准差，以及你最不确定的一题。